# Trial Notebook on Adaptive MH

In [1]:
import os
import gdown
import numpy as np
import torch
from tqdm import tqdm
from scipy.special import logsumexp
import pickle
from time import perf_counter

from torch.distributions.multivariate_normal import MultivariateNormal

from podcnf.DataGenerationLinearElasticity import *
from podcnf.NFmodel import NormalizingFlow
from podcnf.roms import PODcnf, TorchScaler

In [2]:
if torch.cuda.is_available():
        device = torch.device("cuda")
        torch.cuda.manual_seed_all(SEED)
        torch.backends.cudnn.benchmark = True
        scaler = torch.amp.GradScaler(enabled=True)
else:
    device = torch.device("cpu")
    scaler = torch.amp.GradScaler(enabled=False)

In [3]:
# {'learning_rate': 0.001, 'num_flows': 16, 'hidden_size': 256, 'hidden_depth': 2, 'weight_decacy': 1e-05}
os.makedirs('../results/elastic', exist_ok=True)
target_folder = os.path.join("..", "results/elastic")
MODEL_NAME = os.path.join(target_folder, 'MODEL_64_NEW.pth')
gdown.download(id = "1Mv9opjkEMDQaLBQx07Fqvh_ItyefLsCl", quiet=True, output = MODEL_NAME)
loaded_model = torch.load(MODEL_NAME, map_location=device)

In [4]:
# Upload scaler for algorithm
with open("../results/elastic/c_scaler.pkl", "rb") as file:
    c_scal = pickle.load(file)

with open("../results/elastic/mu_scaler.pkl", "rb") as file:
    mu_scal = pickle.load(file)

In [5]:
mu_scaler = TorchScaler(mu_scal.mean_, mu_scal.scale_, device)
c_scaler = TorchScaler(c_scal.mean_, c_scal.scale_, device)

In [6]:
dim_x = mu_scal.n_features_in_
dim_y = c_scal.n_features_in_

In [7]:
# Linear model
num_flows = 16
hidden_size = 256
hidden_depth = 2

flow = NormalizingFlow(dim_x, dim_y, num_flows, hidden_size, hidden_depth, device).to(device)
flow.load_state_dict(loaded_model)

<All keys matched successfully>

In [8]:
V = torch.load("../results/elastic/V_POD_matrix.pt", weights_only=True)

In [9]:
podcnf = PODcnf(V, flow, mu_scaler, c_scaler)

In [10]:
Nh = V.shape[0]

In [11]:
num_sensors = 31
sur = [0,1] # top-left already added
for j in range(num_sensors-1):
    xy = np.array([np.pow(j+2,2)+j, np.pow(j+2,2)+j+1])
    sur.extend(xy)
surface_idx = np.array(sur)

In [12]:
# P = torch.zeros([Nh,1]).to(device)
# P[surface_idx] = int(1)
# P[:20]

In [13]:
V_sensors = V[surface_idx, :]
print(V_sensors.shape)

torch.Size([62, 20])


In [14]:
Q = lambda c: c @ V_sensors.T

In [15]:
def adaptive_metropolis_hastings(
    Gen, Q, mu_0, u_obs, 
    N, bounds, device,
    temperature=1.0, C_0=None, n_0=100, epsilon=1e-6, s_d=None,
    nrep=100, h=0.1
):

    # Initialization
    mu_n = mu_0.clone().detach()
    d = len(mu_n)
    mu_n_scaled = mu_scaler.transform(mu_n.reshape(1, -1))

    def compute_log_likelihood(mu_t):
        with torch.no_grad():

            c_samples = Gen.sample_latent_same_mu(mu_n_scaled, nrep)

            # Projection on the sensors
            # [N_gen, 20] @ [20, 62] -> [N_gen, 62]
            u_sensor_sample = Q(c_samples)

            diff = u_sensor_sample - u_obs.reshape(1, -1)
            dj2 = diff.pow(2).sum(dim=1) # [N_gen]

            # LogSumExp per stabilità (KDE Likelihood)
            # log( sum(exp(-d^2 / 2h^2)) ) - log(N)
            log_pi = torch.logsumexp(-dj2 / (2 * h**2), dim=0) - np.log(nrep)

            return log_pi.item()

    # Initial value for the Log-likelihood
    log_pi_n = compute_log_likelihood(mu_n_scaled)

    chain = []
    accepted_count = 0

    # Covariance
    if C_0 is None:
        C_n = torch.eye(d, dtype=torch.float32, device=device) * 1e-5
    else:
        C_n = torch.tensor(C_0, dtype=torch.float32, device=device)

    mu_bar_n = mu_n.clone().detach()

    zero_mean = torch.zeros(d, dtype=torch.float32, device=device)
    eye_d = torch.eye(d, dtype=torch.float32, device=device)

    # Scaling factor
    if s_d is None:
        scaling_val = (2.38**2) / d
    else:
        scaling_val = s_d

    # MCMC LOOP
    for n in tqdm(range(1, N + 1), desc="Adaptive MH"):

        # Proposal Covariance once n>n_0
        if n <= n_0:
            proposal_cov = C_n
        else:
            proposal_cov = scaling_val * C_n + epsilon * torch.eye(d)

        proposal_cov = (proposal_cov + proposal_cov.T) / 2.0
        
        proposal_dist = MultivariateNormal(zero_mean, proposal_cov)
        perturbation = proposal_dist.sample()
        
        Y = mu_n + perturbation        

        # Check Prior (Bounds)
        if (Y[0] < bounds['m_min'] or Y[0] > bounds['m_max'] or
            Y[1] < bounds['d_min'] or Y[1] > bounds['d_max']):
            chain.append(mu_n)
            mu_next = mu_n
        else:
            # Compute the likelihood for the candidate
            Y_scaled = mu_scaler.transform(Y.reshape(1, -1))

            log_pi_Y = compute_log_likelihood(Y_scaled)

            # Acceptance ratio
            log_alpha = (log_pi_Y - log_pi_n) / temperature

            if np.log(np.random.rand()) < log_alpha:
                mu_n = Y
                log_pi_n = log_pi_Y
                accepted_count += 1

            chain.append(mu_n)
            mu_next = mu_n

        # Updating the covariance
        if n > n_0:
            mu_bar_prev = mu_bar_n.clone().detach()
            mu_bar_n = (n * mu_bar_prev + mu_next) / (n + 1)

            dt = (mu_next - mu_bar_prev).reshape(-1, 1)
            term_update = dt @ dt.T
            C_n = ((n - 1) / n) * C_n + (scaling_val / n) * (term_update * (n / (n + 1)) + epsilon * torch.eye(d))

    chain = np.array(chain)
    acc_rate = accepted_count / N
    print(f"\nAcceptance Rate: {acc_rate:.2%}")
    return chain, C_n

In [16]:
test_idx = 6179 # np.random.randint(n_val, n_samples)
print(test_idx)

6179


In [17]:
# mu and c for the initialization of the model
reduced_dataset = torch.load("../data/elastic_data_reduced_6400.pt" , weights_only=True)
mu = reduced_dataset['mu']
c = reduced_dataset['c']

In [18]:
full_data = torch.load("../data/data_linear_density_2pi.pt")
u = full_data['u_data']

In [19]:
u_surface_sensor = u[:, surface_idx] # selezione direttamente i sensori sulla superficie
print(u_surface_sensor.shape)

torch.Size([6400, 62])


In [21]:
bounds = {
    'm_min': 1.0, 'm_max': 2.0,
    'd_min': 0.05, 'd_max': 0.25
}

mu_true_phys = mu[test_idx].numpy()
u_obs = u_surface_sensor[test_idx]

mu_0 = torch.tensor(
    [np.random.rand() + 1, np.random.rand() * 0.1 + 0.15], 
    dtype=torch.float32, 
    device=device
)

print(f"Index: {test_idx} - True Mass={mu_true_phys[0]:.4f}, True Delta={mu_true_phys[1]:.4f}")
print(f"Initial guess: {mu_0}")

print("\n--- EXPLORATION ---")
initial_cov_expl = [[0.001, 0], [0, 0.001]]

t0 = perf_counter()

chain_exploration, cov_learned = adaptive_metropolis_hastings(
    podcnf, Q, mu_0, u_obs,
    N=10000, bounds=bounds, device=device,
    temperature=5.0, C_0=initial_cov_expl, n_0=500, s_d=2.4,
    nrep=100, h=0.1
)

t_exp = perf_counter() - t0

Index: 6179 - True Mass=1.5729, True Delta=0.1666
Initial guess: tensor([1.4296, 0.2446])

--- EXPLORATION ---


Adaptive MH: 100%|██████████| 10000/10000 [02:35<00:00, 64.34it/s]


Acceptance Rate: 26.35%


In [ ]:
best_guess = torch.tensor(chain_exploration[-1])
print(f"Best Guess after exploration: {best_guess}")

print("\n--- REFINEMENT ---")
cov_for_refinement = cov_learned + np.eye(2) * 1e-6
cov_for_refinement = cov_for_refinement.clone().detach().tolist()

t1 = perf_counter()

chain_refined, _ = adaptive_metropolis_hastings(
    podcnf, Q, best_guess, u_obs,
    N=30000, bounds=bounds, device=device,
    temperature=1.0, C_0=cov_for_refinement, n_0=0, s_d=0.35,
    nrep=200, h=0.03
)

t_ref = perf_counter() - t1

/tmp/ipykernel_2691/2046578092.py:5: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  cov_for_refinement = cov_learned + np.eye(2) * 1e-6


Best Guess after exploration: tensor([1.2437, 0.2395])

--- REFINEMENT ---


Adaptive MH:  65%|██████▍   | 19430/30000 [34:35<35:25,  4.97it/s]  